<a href="https://colab.research.google.com/github/Nakib-Nasrullah/Heart_disease/blob/main/Transformer_Based_ECG_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install wfdb pandas numpy scikit-learn tensorflow matplotlib seaborn
import os
import wfdb
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from tensorflow.keras.layers import *
from tensorflow.keras.models import Model


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 62.1 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.


# **Download Dataset**

In [2]:
DATA_DIR = "mitdb"

if not os.path.exists(DATA_DIR):
    wfdb.dl_database('mitdb', dl_dir=DATA_DIR)

print("Dataset ready!")

Generating record list for: 100
Generating record list for: 101
Generating record list for: 102
Generating record list for: 103
Generating record list for: 104
Generating record list for: 105
Generating record list for: 106
Generating record list for: 107
Generating record list for: 108
Generating record list for: 109
Generating record list for: 111
Generating record list for: 112
Generating record list for: 113
Generating record list for: 114
Generating record list for: 115
Generating record list for: 116
Generating record list for: 117
Generating record list for: 118
Generating record list for: 119
Generating record list for: 121
Generating record list for: 122
Generating record list for: 123
Generating record list for: 124
Generating record list for: 200
Generating record list for: 201
Generating record list for: 202
Generating record list for: 203
Generating record list for: 205
Generating record list for: 207
Generating record list for: 208
Generating record list for: 209
Generati

# **Beat Extraction**

In [4]:
WINDOW = 187
HALF = WINDOW // 2

label_map = {
    'N': 0, 'L': 0, 'R': 0, 'e': 0, 'j': 0,
    'A': 1, 'a': 1, 'J': 1, 'S': 1,
    'V': 2, 'E': 2,
    'F': 3
}

beats = []

records = sorted([f.split('.')[0] for f in os.listdir(DATA_DIR) if f.endswith('.dat')])

for record in records:
    try:
        signal, _ = wfdb.rdsamp(os.path.join(DATA_DIR, record))
        ann = wfdb.rdann(os.path.join(DATA_DIR, record), 'atr')
    except:
        continue

    ecg = signal[:, 0]

    for r, sym in zip(ann.sample, ann.symbol):
        if sym not in label_map:
            continue

        if r - HALF < 0 or r + HALF >= len(ecg):
            continue

        beat = ecg[r-HALF:r+HALF+1]
        beats.append([record] + beat.tolist() + [label_map[sym]])

columns = ["record_id"] + [f"f{i}" for i in range(WINDOW)] + ["label"]
df = pd.DataFrame(beats, columns=columns)

df.to_csv("mitbih_patient_level.csv", index=False)
print("Dataset created:", df.shape)


Dataset created: (101426, 189)


# **Patient-Level Split**

In [5]:
df = pd.read_csv("mitbih_patient_level.csv")

patients = df['record_id'].unique()

trainval_patients, test_patients = train_test_split(
    patients, test_size=0.20, random_state=42
)

train_patients, val_patients = train_test_split(
    trainval_patients, test_size=0.10, random_state=42
)

train_df = df[df['record_id'].isin(train_patients)]
val_df   = df[df['record_id'].isin(val_patients)]
test_df  = df[df['record_id'].isin(test_patients)]

print("No patient overlap ✔")


No patient overlap ✔


# **Data Preparation**

In [6]:
def prepare(df):
    X = df.iloc[:, 1:-1].values
    y = df['label'].values.astype(int)

    X = (X - X.mean(axis=1, keepdims=True)) / \
        (X.std(axis=1, keepdims=True) + 1e-8)

    return X.reshape(-1, 187, 1), y

X_train, y_train = prepare(train_df)
X_val, y_val     = prepare(val_df)
X_test, y_test   = prepare(test_df)

print(X_train.shape, X_val.shape, X_test.shape)


(72737, 187, 1) (8245, 187, 1) (20444, 187, 1)
